# Week 3 — 해석 고정 (ablation · 유의성 · axis 연결 · 최종 산출물)

> 2주차 sweep 결과(`sweep_*_*.csv`)와 1주차 자산을 받아 **해석을 굳힌다.**
> 새 분석은 추가하지 않는다 — 기존 결과를 ablation·유의성·axis 그림과 연결만 한다.

### 입력
- `results/sweep_coarse_*.csv`, `results/sweep_fine_*.csv`, `results/main_table_*.csv`
- `results/v_AB.npy … v_harm.npy`, `results/probe.pkl`, `results/b0_baseline.json`, `results/fn_subset_*.npy`
- `data/eval/eval_latent_v2.csv`, `data/eval/eval_toxigen_v1.csv`

### 산출물
- `results/ablation_*.csv` — α=0 / 부호반전(−α) / 첫토큰 / 전토큰
- `results/significance_*.json` — McNemar p + ΔF1 bootstrap CI
- `results/graphA_axis_*.png` — best layer vs harm-dominant 레이어 겹쳐보기
- 한계/future work 단락 템플릿

## 0. 설정 & 로드

In [ ]:
import os, json, re, sys
import numpy as np, pandas as pd, torch, joblib
from sklearn.metrics import f1_score
from scipy.stats import binomtest
import matplotlib.pyplot as plt

class CFG:
    MODEL="meta-llama/Llama-3.2-3B"; OUT="results"; DATA="data/eval"; SRC="src/eval"
    BATCH=32; MAX_LEN=128; SEED=42
    HARM_LAYERS=[4,5,9,10,11,13,14,18,19,20,21,22,23,24,25,26,27]   # 0528 axis attribution
device="cuda" if torch.cuda.is_available() else "cpu"; print("device:", device)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tok=AutoTokenizer.from_pretrained(CFG.MODEL)
if tok.pad_token is None: tok.pad_token=tok.eos_token
tok.padding_side="left"
model=AutoModelForCausalLM.from_pretrained(CFG.MODEL, torch_dtype=torch.float16).to(device).eval()
model.config.output_hidden_states=False
N_LAYERS=model.config.num_hidden_layers

In [ ]:
sys.path.insert(0, CFG.SRC); from metrics import evaluate, evaluate_by_group, HATE
pb=joblib.load(os.path.join(CFG.OUT,"probe.pkl")); scaler,clf=pb["scaler"],pb["clf"]
b0=json.load(open(os.path.join(CFG.OUT,"b0_baseline.json")))

b0m = {
    "eval_latent_v2": b0["results"]["eval_latent_v2"],
    "eval_toxigen_v1": b0["results"]["eval_toxigen_v1"],
}

def load_eval(path, group=False):
    df=pd.read_csv(path)
    if df["label"].dtype==object:
        df["label"]=df["label"].astype(str).str.strip().str.lower().map({"hate":1,"non-hate":0})
    df["label"]=df["label"].astype(int)
    g=df["target_group"].tolist() if (group and "target_group" in df) else None
    return df["text"].tolist(), df["label"].to_numpy(), g

EV = {
    "eval_latent_v2": load_eval(os.path.join(CFG.DATA, "eval_latent_v2.csv")),
    "eval_toxigen_v1": load_eval(os.path.join(CFG.DATA, "eval_toxigen_v1.csv"), group=True),
}

FN = {
    "eval_latent_v2": np.load(os.path.join(CFG.OUT, "fn_subset_eval_latent_v2.npy")),
    "eval_toxigen_v1": np.load(os.path.join(CFG.OUT, "fn_subset_eval_toxigen_v1.npy")),
}

VECTORS={n: np.load(os.path.join(CFG.OUT,f"{n}.npy")) for n in ["v_AB","v_AC","v_random","v_harm"]}

def load_sweep(tag):
    c=pd.read_csv(os.path.join(CFG.OUT,f"sweep_coarse_{tag}.csv"))
    fp=os.path.join(CFG.OUT,f"sweep_fine_{tag}.csv")
    return pd.concat([c, pd.read_csv(fp)], ignore_index=True) if os.path.exists(fp) else c
SW={tag: load_sweep(tag) for tag in EV}
print("loaded. B0:", {k:round(v["macro_f1"],4) for k,v in b0m.items()})

## 1. 공용 함수 (위치별 hook 포함)

In [ ]:
_ABL={}   # 첫토큰 주입용 per-batch 인덱스

class AblHook:
    '''mode: last(기본) / first(첫 실제 토큰) / all(전 토큰) / none(주입 안 함)'''
    def __init__(self, vec, alpha, mode="last"):
        self.v=torch.tensor(vec, dtype=torch.float16, device=device); self.a=float(alpha); self.mode=mode; self.h=None
    def _fn(self, m, i, o):
        h = o[0] if isinstance(o, tuple) else o
        if self.mode=="last":
            h[:, -1, :] = h[:, -1, :] + self.a*self.v
        elif self.mode=="first":
            fi=_ABL["first"]; B=h.shape[0]
            ar=torch.arange(B, device=h.device)
            h[ar, fi, :] = h[ar, fi, :] + self.a*self.v
        elif self.mode=="all":
            h[:, :, :] = h + self.a*self.v
        return o
    def attach(self, layer): self.h=layer.register_forward_hook(self._fn); return self
    def detach(self):
        if self.h is not None: self.h.remove(); self.h=None

def prebatch(texts):
    texts=[str(t) for t in texts]
    order=sorted(range(len(texts)), key=lambda i: len(texts[i])); inv=np.argsort(order)
    B=[tok([texts[i] for i in order[s:s+CFG.BATCH]], return_tensors="pt", padding=True,
           truncation=True, max_length=CFG.MAX_LEN) for s in range(0,len(order),CFG.BATCH)]
    return B, inv

@torch.inference_mode()
def extract(batches, inv, vec=None, L=None, alpha=0.0, mode="none"):
    hk = AblHook(vec[L], alpha, mode).attach(model.model.layers[L]) if mode!="none" else None
    out=[]
    try:
        for enc in batches:
            enc={k:v.to(device) for k,v in enc.items()}
            if mode=="first": _ABL["first"]=(enc["attention_mask"]==0).sum(1)  # left-pad: 선행 pad 수 = 첫 실제 토큰 idx
            out.append(model.model(**enc, use_cache=False, output_hidden_states=False).last_hidden_state[:, -1, :].float().cpu().numpy())
    finally:
        if hk: hk.detach()
    return np.concatenate(out,0)[inv]

def predict(X): return clf.predict(scaler.transform(X))
def recovery(pred, fn): return 0.0 if len(fn)==0 else float((pred[fn]==HATE).mean())

def mcnemar(y, p0, p1):
    '''B0 vs E1 짝지은 정오 비교. b=B0만 맞음, c=E1만 맞음.'''
    c0=(p0==y); c1=(p1==y)
    b=int(np.sum(c0 & ~c1)); c=int(np.sum(~c0 & c1))
    p=binomtest(min(b,c), b+c, 0.5).pvalue if (b+c)>0 else 1.0
    return dict(b_only_B0=b, c_only_E1=c, p_value=float(p))

def boot_delta_f1(y, p0, p1, n=2000, seed=CFG.SEED):
    rng=np.random.RandomState(seed); idx=np.arange(len(y)); d=[]
    for _ in range(n):
        s=rng.choice(idx, len(idx), replace=True)
        d.append(f1_score(y[s],p1[s],average="macro")-f1_score(y[s],p0[s],average="macro"))
    lo,med,hi=np.percentile(d,[2.5,50,97.5]); return dict(lo=float(lo),med=float(med),hi=float(hi))

## 2. Best 셋업 확정 (coarse+fine)

In [ ]:
def best_of(df, name):
    sub=df[df.vector==name]; b=sub.loc[sub.macro_f1.idxmax()]
    return int(b.layer), float(b.alpha), float(b.macro_f1)
BEST={tag:{n:best_of(SW[tag],n) for n in VECTORS} for tag in SW}
for tag in BEST:
    print(f"[{tag}] B0={b0m[tag]['macro_f1']:.4f}")
    for n,(L,a,f) in BEST[tag].items():
        print(f"   {n:9s} L={L:2d} α={a:>5} F1={f:.4f} (Δ{f-b0m[tag]['macro_f1']:+.4f})")

## 3. Ablation (메인 셋업 기준)
- **α=0**: hook은 걸되 0 곱 → B0와 같아야 함(hook 자체 무해 확인).
- **부호반전(−α)**: 떨어지면 "미는 방향이 맞다"는 증거.
- **첫토큰 / 전토큰 주입**: 마지막 토큰 위치가 중요한지.

In [ ]:
def run_ablation(tag):
    texts, labels, _ = EV[tag]; fn=FN[tag]; base=b0m[tag]["macro_f1"]
    batches, inv = prebatch(texts); rows=[]
    for name in ["v_AB","v_AC"]:
        L,a,_=BEST[tag][name]
        settings=[("main (last,+α)",a,"last"),("α=0",0.0,"last"),("sign flip (−α)",-a,"last"),
                  ("first-token",a,"first"),("all-token",a,"all")]
        for slabel, al, mode in settings:
            X=extract(batches, inv, VECTORS[name], L, al, mode)
            pred=predict(X); m=evaluate(pred, labels)
            rows.append(dict(vector=name, L=L, setting=slabel,
                             macro_f1=round(m["macro_f1"],4), d_f1=round(m["macro_f1"]-base,4),
                             fn_rec=round(recovery(pred,fn),4)))
    t=pd.DataFrame(rows); t.to_csv(os.path.join(CFG.OUT,f"ablation_{tag}.csv"), index=False)
    print(f"\n===== ABLATION [{tag}]  (B0 F1={base:.4f}) =====\n{t.to_string(index=False)}")
    return t

A_latent = run_ablation("eval_latent_v2")
A_tg = run_ablation("eval_toxigen_v1")

## 4. 유의성 검정 (B0 vs E1 v_AB)
McNemar(짝지은 정오)와 ΔmacroF1 bootstrap 95% CI. "의미 있게 이긴다"의 근거.

In [ ]:
sig={}
for tag in EV:
    texts, labels, _ = EV[tag]; batches, inv = prebatch(texts)
    p0 = predict(extract(batches, inv, mode="none"))          # B0 재계산 (저장값과 교차확인)
    L,a,_ = BEST[tag]["v_AB"]
    p1 = predict(extract(batches, inv, VECTORS["v_AB"], L, a, "last"))
    mc = mcnemar(labels, p0, p1); bt = boot_delta_f1(labels, p0, p1)
    f0=f1_score(labels,p0,average="macro"); f1v=f1_score(labels,p1,average="macro")
    sig[tag]=dict(B0_macro_f1=round(f0,4), E1_macro_f1=round(f1v,4),
                  delta=round(f1v-f0,4), mcnemar=mc, bootstrap_delta_ci=bt,
                  best=dict(layer=L, alpha=a))
    print(f"[{tag}] B0={f0:.4f} → E1={f1v:.4f} (Δ{f1v-f0:+.4f}) | "
          f"McNemar p={mc['p_value']:.2e} (b={mc['b_only_B0']}, c={mc['c_only_E1']}) | "
          f"ΔF1 95%CI [{bt['lo']:+.4f}, {bt['hi']:+.4f}]")
json.dump(sig, open(os.path.join(CFG.OUT,"significance.json"),"w"), indent=2)

## 5. Axis attribution 연결
best layer가 0528에서 harm-dominant로 찍힌 레이어 안에 드는지. 새 분석이 아니라 *일치 확인*만.

In [ ]:
# ── 0528 mean-pool 리스트 대신 last-token harm 축 곡선 재산출 ──
CELL_C = "cell_c_test_final.csv"          # 본인 경로
_dfc = pd.read_csv(CELL_C).dropna(subset=["cell_a","cell_c_modified"])
A_texts = _dfc["cell_a"].tolist(); C_texts = _dfc["cell_c_modified"].tolist()

@torch.inference_mode()
def extract_all_layers(texts):
    out=[]
    for s in range(0, len(texts), CFG.BATCH):
        enc = tok([str(t) for t in texts[s:s+CFG.BATCH]], return_tensors="pt", padding=True,
                  truncation=True, max_length=CFG.MAX_LEN).to(device)
        hs = model.model(**enc, use_cache=False, output_hidden_states=True).hidden_states
        out.append(np.stack([h[:, -1, :].float().cpu().numpy() for h in hs], axis=1))
    return np.concatenate(out, 0)         # (n, L+1, H)

hA = extract_all_layers(A_texts); hC = extract_all_layers(C_texts)
vh = VECTORS["v_harm"]                                  # (L+1, H) last-token 단위
pa = np.einsum("nlh,lh->nl", hA, vh)                    # A를 harm축에 사영
pc = np.einsum("nlh,lh->nl", hC, vh)
pooled = np.concatenate([pa, pc], 0).std(0) + 1e-8
harm_gap_z = (pa.mean(0) - pc.mean(0)) / pooled         # (L+1,) — |z| 클수록 harm-dominant
np.save(os.path.join(CFG.OUT, "harm_gap_z_lasttoken.npy"), harm_gap_z)

order = np.argsort(-np.abs(harm_gap_z))
print("last-token harm |z| 상위 레이어:", order[:10].tolist())
print("0528 mean-pool 리스트       :", CFG.HARM_LAYERS)
for tag in BEST:
    for n in ["v_AB","v_AC"]:
        L = BEST[tag][n][0]
        rank = int(np.where(order==L)[0][0]) + 1
        print(f"[{tag}] {n} best L={L} | last-token harm |z| {rank}위 (z={harm_gap_z[L]:+.3f})")

def graphA_axis(tag):
    df = SW[tag]; base = b0m[tag]["macro_f1"]
    fig, ax = plt.subplots(figsize=(11,5))
    for n,lab in [("v_AB","E1 v_AB"),("v_AC","E2 v_AC"),("v_random","B1 rand"),("v_harm","B2 harm")]:
        sub=df[df.vector==n]; ba=sub.loc[sub.macro_f1.idxmax(),"alpha"]
        s=sub[sub.alpha==ba].sort_values("layer")
        ax.plot(s.layer, s.macro_f1, marker="o", ms=3, label=f"{lab} (α={ba})")
    ax.axhline(base, color="k", ls="--", label="B0")
    ax.set_xlabel("layer"); ax.set_ylabel("macro F1"); ax.legend(fontsize=8, loc="upper left")
    ax2 = ax.twinx()                                    # 이중축: harm 축 강도(last-token)
    xs = range(len(harm_gap_z))
    ax2.plot(xs, np.abs(harm_gap_z), color="orange", lw=1.6, ls=":", alpha=0.8)
    ax2.fill_between(xs, np.abs(harm_gap_z), color="orange", alpha=0.08)
    ax2.set_ylabel("harm-axis |A−C z| (last-token)", color="orange")
    ax.set_title(f"Graph A + harm-axis(last-token) [{tag}]")
    plt.tight_layout(); plt.savefig(os.path.join(CFG.OUT, f"graphA_axis_{tag}.png"), dpi=150); plt.show()

graphA_axis("eval_latent_v2")
graphA_axis("eval_toxigen_v1")

In [ ]:
def graphB(tag):
    df = SW[tag]; base = b0m[tag]["macro_f1"]
    fig, axes = plt.subplots(1, 2, figsize=(12,5))
    for ax, n, title in zip(axes, ["v_AB","v_AC"], ["E1 v_AB (target)","E2 v_AC (cue)"]):
        sub = df[df.vector==n]; L = int(sub.loc[sub.macro_f1.idxmax(), "layer"])
        s = sub[(sub.layer==L) & (sub.alpha > 0)].sort_values("alpha")   # 양수 α만 (부호반전은 ablation)
        ax.plot(s.alpha, s.macro_f1, marker="o")
        ax.axhline(base, color="k", ls="--", label="B0")
        ax.set_title(f"{title}\nbest layer {L} [{tag}]")
        ax.set_xlabel("steering strength α"); ax.set_ylabel("macro F1"); ax.legend()
    plt.tight_layout(); plt.savefig(os.path.join(CFG.OUT, f"graphB_{tag}.png"), dpi=150); plt.show()

graphB("eval_latent_v2")
graphB("eval_toxigen_v1")

## 6. 최종 표 (본문용)

In [ ]:
def final_table(tag):
    df = SW[tag]; base = b0m[tag]
    label = {"v_random":"B1 Random","v_harm":"B2 v_harm","v_AB":"E1 v_AB (target)","v_AC":"E2 v_AC (cue)"}
    rows = [dict(Setup="B0 No steering", layer="—", alpha="—",
                 macro_f1=round(base["macro_f1"],4), hate_recall=round(base["hate_recall"],4),
                 fn_recovery="—", d_f1=0.0)]
    for n in ["v_random","v_harm","v_AB","v_AC"]:
        L, a, f = best_of(df, n)
        r = df[(df.vector==n)&(df.layer==L)&(df.alpha==a)].iloc[0]
        rows.append(dict(Setup=label[n], layer=L, alpha=a,
                         macro_f1=round(f,4), hate_recall=round(r.hate_recall,4),
                         fn_recovery=round(r.fn_recovery,4),          # ← 추가
                         d_f1=round(f-base["macro_f1"],4)))
    t = pd.DataFrame(rows); t.to_csv(os.path.join(CFG.OUT, f"final_table_{tag}.csv"), index=False)
    print(f"\n===== FINAL [{tag}] =====\n{t.to_string(index=False)}"); return t

final_table("eval_latent_v2")
final_table("eval_toxigen_v1")

## 7. 한계 / future work — 단락 템플릿

아래 빈칸을 결과로 채워 본문 마지막 단락에 넣는다.

> 본 연구는 **단일 모델(Llama-3.2-3B)·추론 시점 개입** 수준의 인과 주장에 한정된다.
> 스티어링 벡터는 minimal-pair(Cell A/B/C, N=___)의 last-token 표상 차이로 정의했고,
> 평가는 출처가 다른 두 셋(Latent Hatred n=2000, ToxiGen-HumanVal n=____)에서 수행했다.
> probe는 HateXplain explicit 중심으로 학습되어 implicit hate에 대한 일반화 여지가 컸으며,
> 이 갭을 target 축 벡터(v_AB)가 ΔF1=____(95%CI[__,__], McNemar p=__)만큼 메웠다.
> 한계: (1) 단일 모델·단일 probe, (2) harm-dominant 레이어는 mean-pool 공간 산출이라 last-token best layer와 다를 수 있음(각주),
> (3) v_sent·v_harm 직교성 등 표상 분리 분석은 future work로 남김.
> 확장: 다른 규모 모델, 다국어 hate, attention 수준 기여 분석.